# Galaxy-Pick — PJ1 Analysis

**Problem.** A shopper walking into a store faces ~36 current Galaxy models and no good way to choose. Spec sheets don't answer *"which one is right for me?"* — that depends on what the shopper actually cares about.

**Approach.** We turn raw specs into four interpretable 0–10 scores (**camera, performance, battery, value**), then rank phones with a **Weighted Sum Model (WSM)**: each shopper persona supplies a weight vector over those four factors, and every phone gets a `match_score`. It is fully deterministic and every recommendation can be explained in one line — no black box.

**Honesty about the data (Ch. 4).** The dataset is *curated*, not scraped:
- **2024–2025 rows (27) are real** specs, human-verified.
- **2026 rows (9) are `spec_source=mock`** — plausible projections seeded with Gemini and then human-corrected. They are **projections, not facts**, and are labelled as such everywhere they appear.

Prices are representative base MRP in **INR (₹)**.

> All logic lives in `src/recommender/` and is imported here — the notebook and the Streamlit app run the **same engine**, so they can never disagree.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.recommender import data, explain, personas, scoring, wsm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (7, 3.5)
plt.rcParams["axes.spines.top"] = plt.rcParams["axes.spines.right"] = False

SAMSUNG_BLUE = "#1428A0"
print("engine imported from", ROOT / "src" / "recommender")

## 1. Load the catalog

`data.load_phones()` reads `data/processed/phones.csv` **and validates it** — the row contract (2024–2026 only, every 2026 row flagged `mock`, all scores within [0,10], no nulls in required columns) is enforced on load, so a broken dataset fails loudly here rather than silently skewing a recommendation.

In [ ]:
df = data.load_phones()

print("shape:", df.shape)
print("provenance:", data.provenance(df))
df.head()

In [ ]:
df.describe().round(1)

## 2. Exploratory data analysis — three findings

Each finding below is backed by a number and a chart, and each one **changed a downstream design decision**.

### Finding 1 — the segments don't overlap on price, and "photography" is a luxury tier

Samsung's marketing segments map onto cleanly separated price bands, with one striking gap: **every phone in the `photography` segment costs ≥ ₹1,39,999**, while the entire `budget` segment tops out at ₹21,999. There is no such thing as an affordable "photography" phone in this catalog.

**Why it matters:** a persona who wants a great camera on a modest budget cannot be served the *marketing* answer. The recommender has to rank on measured capability within their budget instead — which is exactly what the WSM does.

In [ ]:
price_by_segment = df.groupby("segment")["price_inr"].agg(["min", "median", "max", "count"]).sort_values("median")
display(price_by_segment)

print(f"photography segment starts at ₹{df[df.segment=='photography'].price_inr.min():,}")
print(f"budget segment tops out at    ₹{df[df.segment=='budget'].price_inr.max():,}")
print(f"→ gap between the two bands:  ₹{df[df.segment=='photography'].price_inr.min() - df[df.segment=='budget'].price_inr.max():,}")

ax = price_by_segment[["min", "max"]].plot(kind="barh", color=["#9DB2F2", SAMSUNG_BLUE])
ax.set_xlabel("price (₹)")
ax.set_ylabel("")
ax.set_title("Finding 1 — price bands by segment (min vs max)")
plt.tight_layout(); plt.show()

### Finding 2 — the surprise: cheap phones have the *biggest* batteries

Price buys almost everything in a phone — except battery capacity, where the relationship **inverts**. The `budget` segment averages the largest cells in the catalog, comfortably beating `business` phones that cost up to 10× more. Budget M/F models ship 6000 mAh; flagships hover near 4000–5000 mAh because they are thinner and spend their cost budget on silicon, optics and displays.

**Why it matters:** battery is the one factor where a budget-conscious persona is not compromising at all. It's also why `battery_score` and `value_score` reinforce each other for the student persona — and why our EDA needed to check for exactly this kind of correlation rather than assume "expensive = better on every axis".

In [ ]:
battery_by_segment = df.groupby("segment")["battery_mah"].mean().sort_values(ascending=False)
display(battery_by_segment.round(0))

corr = df["price_inr"].corr(df["battery_mah"])
print(f"correlation(price, battery_mah) = {corr:+.2f}   ← negative: paying more buys LESS battery")
print(f"budget avg   {df[df.segment=='budget'].battery_mah.mean():,.0f} mAh")
print(f"business avg {df[df.segment=='business'].battery_mah.mean():,.0f} mAh")

fig, ax = plt.subplots()
ax.scatter(df["price_inr"], df["battery_mah"], color=SAMSUNG_BLUE, alpha=0.75)
ax.set_xlabel("price (₹)"); ax.set_ylabel("battery (mAh)")
ax.set_title(f"Finding 2 — price vs battery, r = {corr:+.2f}")
plt.tight_layout(); plt.show()

### Finding 3 — megapixels are useless as a camera signal here

`rear_camera_mp` takes **only two distinct values across all 36 phones**: 50 and 200. 31 of 36 are 50MP — and every 200MP phone costs ≥ ₹1,39,999. So megapixel count carries almost no information: it says a ₹12,499 Galaxy F15 and a ₹87,559 Galaxy S24 have *the same camera*, and below ₹1.4 lakh it cannot tell any two phones apart at all.

**Why it matters — this finding changed the model.** Our first `camera_score` scored mostly on megapixels. The result was a WSM that rated the **cheapest phone in the catalog as the best photography phone**, because a criterion that never varies cannot influence a ranking, so `value_score` decided every persona by default. We rebuilt `camera_score` around the **series optics tier** (sensor size, OIS, zoom, ISP — the things that actually differ, and which track the series) with megapixels demoted to a small bonus. See §4.

In [ ]:
print("distinct rear_camera_mp values across the whole catalog:", sorted(df["rear_camera_mp"].unique()))
display(df["rear_camera_mp"].value_counts().rename("phones"))
print(f"cheapest 200MP phone: ₹{df[df.rear_camera_mp==200].price_inr.min():,}")
print(f"phones under ₹80,000 that are NOT 50MP: {(df[df.price_inr<=80000].rear_camera_mp != 50).sum()}  ← none")

# How much does each raw spec actually vary between segments? (coefficient of variation
# of the segment means — higher = more discriminating)
specs = ["price_inr", "rear_camera_mp", "charging_w", "ram_gb", "refresh_rate_hz", "battery_mah", "screen_size_inch"]
spread = pd.Series({c: df.groupby("segment")[c].mean().std() / df.groupby("segment")[c].mean().mean() for c in specs})
spread = spread.sort_values(ascending=False).rename("between-segment variation (CV)")
display(spread.round(3))
print("→ screen size and battery barely vary between segments; price and MP vary most.")

## 3. Feature engineering — raw specs → four 0–10 scores

We import `scoring.py` rather than re-implementing it, so the notebook, the tests and the app are provably scoring phones the same way.

The scores are **transparent heuristics, not benchmarks**. That is a deliberate trade: we can show a shopper exactly why a phone ranked where it did.

In [ ]:
# Step 6 of the brief asks for the LOGIC, not just the numbers.
print(scoring.LOGIC_SUMMARY)

In [ ]:
# Re-derive the scores from raw specs to prove phones.csv is reproducible from the seed.
seed = pd.read_csv(ROOT / "data/raw/curated_seed.csv")

rebuilt = seed.assign(
    camera_score=[scoring.camera_score(mp, s) for mp, s in zip(seed.rear_camera_mp, seed.series)],
    performance_score=[scoring.performance_score(c, r, hz) for c, r, hz in zip(seed.chipset, seed.ram_gb, seed.refresh_rate_hz)],
    battery_score=[scoring.battery_score(b, w, sc) for b, w, sc in zip(seed.battery_mah, seed.charging_w, seed.screen_size_inch)],
)
rebuilt["value_score"] = scoring.value_score_column(rebuilt)

matches = all((rebuilt[c].values == df[c].values).all() for c in wsm.SCORE_COLS)
print("re-derived scores identical to phones.csv:", matches)

rebuilt.loc[[0, 2, 15], ["model_name", "series", "rear_camera_mp", "chipset", "price_inr"] + wsm.SCORE_COLS]

In [ ]:
# camera_score now separates phones that megapixels alone called identical.
fifty_mp = df[df.rear_camera_mp == 50]
display(fifty_mp.groupby("series")["camera_score"].first().sort_values(ascending=False).rename("camera_score (all 50MP phones)"))

# value_score is emergent — nobody hand-assigned it; it falls out of specs-per-rupee.
display(df.groupby("series")["value_score"].mean().sort_values(ascending=False).round(1).rename("mean value_score"))
print("→ budget F/M dominate value; foldables score ~0. The model discovered the value ladder on its own.")

## 4. Personas

Four shoppers, each a weight vector over the four factors. The weights *are* the model — change them and the ranking changes.

| Persona | Camera | Performance | Battery | Value | Budget |
|---|---|---|---|---|---|
| **Priya** — photography enthusiast | **0.50** | 0.10 | 0.20 | 0.20 | ₹80,000 |
| **Arjun** — mobile gamer | 0.10 | **0.50** | 0.30 | 0.10 | ₹85,000 |
| **Meera** — budget-conscious student | 0.15 | 0.20 | 0.30 | **0.35** | ₹20,000 |
| **Rahul** — business all-rounder | 0.25 | 0.30 | 0.25 | 0.20 | ₹1,40,000 |

In [ ]:
for name, p in personas.PERSONAS.items():
    total = sum(p["weights"].values())
    print(f"{name}\n  {p['story']}\n  weights={p['weights']}  sum={total:.2f}  budget=₹{p['budget_max']:,}")
    assert abs(total - 1.0) < 1e-9, f"{name} weights must sum to 1.0"
print("\n✓ all four weight vectors sum to 1.0 → match_score is guaranteed to land on the same 0–10 scale")

## 5. The Weighted Sum Model

$$\text{match\_score} = \text{camera} \times w_c + \text{performance} \times w_p + \text{battery} \times w_b + \text{value} \times w_v$$

With scores on 0–10 and weights summing to 1, `match_score` is itself on 0–10.

### Worked example — Priya
Weights `0.5 / 0.1 / 0.2 / 0.2`, and a phone scoring camera **9**, performance **6**, battery **7**, value **5**:

$$9(0.5) + 6(0.1) + 7(0.2) + 5(0.2) = 4.5 + 0.6 + 1.4 + 1.0 = \mathbf{7.5}$$

In [ ]:
example = pd.DataFrame([{"camera_score": 9, "performance_score": 6, "battery_score": 7, "value_score": 5}])
weights = {"camera": 0.5, "performance": 0.1, "battery": 0.2, "value": 0.2}

result = float(wsm.match_scores(example, weights).iloc[0])
print("9×0.5 + 6×0.1 + 7×0.2 + 5×0.2 =", result)
assert result == 7.5

### Where our first WSM was wrong (Ch. 4 — guarding the model)

The formula above is correct but **not sufficient**, and our first version of this notebook shipped a WSM that quietly ignored its own weights.

A criterion's real influence in a weighted sum is **weight × spread**, not weight alone. Our four raw scores did not share a spread: `value_score` is min–max scaled and uses the full 0–10 range, while `camera_score` spanned about 3 points. So value swung the ranking by 2.0 points at a weight of 0.20, while camera swung it by only 1.5 at a weight of 0.50 — **value outvoted camera at 2.5× less weight**, and every persona collapsed onto the cheapest phone in the catalog. Priya the photography enthusiast and Meera the budget student got *identical* top-3s.

The fix is standard multi-criteria decision analysis: **normalize the decision matrix** — the alternatives actually being compared — onto a common scale before applying weights. `wsm.normalize_scores()` does this on the in-budget pool, so "weight 0.50" genuinely means "half the decision". Raw scores are still what we *show* the shopper, because "camera 7.4/10" is meaningful while a pool-relative 0.0 is not.

*Lesson for the deck: the arithmetic was never wrong. The **scales** were — and a plausible-looking model was confidently recommending a ₹12,499 phone to a photographer.*

In [ ]:
# The imbalance, quantified — this is the bug that normalization removes.
priya = personas.PERSONAS["Priya — Photography enthusiast"]["weights"]

diag = pd.DataFrame({
    "weight": [priya[f] for f in wsm.FACTORS],
    "raw spread": [round(df[c].max() - df[c].min(), 1) for c in wsm.SCORE_COLS],
}, index=wsm.FACTORS)
diag["influence BEFORE (weight × spread)"] = (diag["weight"] * diag["raw spread"]).round(2)
diag["influence AFTER (weight × 10)"] = (diag["weight"] * 10).round(2)
display(diag)
print("Before: value (0.20) swings 2.00 but camera (0.50) swings only 1.50 → value wins. After: weights decide.")

## 6. Recommendations for each persona

`wsm.recommend()` = filter to budget → normalize the pool → weighted sum → rank → top-3.

In [ ]:
SHOW = ["model_name", "price_inr", "camera_score", "performance_score", "battery_score", "value_score", "match_score", "spec_source"]

for name, p in personas.PERSONAS.items():
    top3 = wsm.recommend(df, p["weights"], budget_max=p["budget_max"], top_n=3)
    close = explain.is_close_call(top3)

    print("=" * 100)
    print(f"{name}   —   budget ₹{p['budget_max']:,}")
    print(f"{p['story']}")
    print("=" * 100)
    display(top3[SHOW].reset_index(drop=True))
    for i, (_, row) in enumerate(top3.iterrows(), 1):
        print(f"  {i}. {explain.reason(row, p['weights'], persona_label=name, close_call=close and i <= 2)}")
    print()

In [ ]:
# Sanity check: the personas must actually disagree, and each must win on what it weights.
winners = {name: wsm.recommend(df, p["weights"], budget_max=p["budget_max"], top_n=1).iloc[0]["model_name"]
           for name, p in personas.PERSONAS.items()}
for name, model in winners.items():
    print(f"{model:20s} ← {name}")
print(f"\ndistinct winners: {len(set(winners.values()))} of {len(winners)} personas")

priya_p = personas.PERSONAS["Priya — Photography enthusiast"]
best_cam = df[df.price_inr <= priya_p["budget_max"]]["camera_score"].max()
got_cam = wsm.recommend(df, priya_p["weights"], budget_max=priya_p["budget_max"], top_n=1).iloc[0]["camera_score"]
print(f"Priya's pick has camera {got_cam}/10; best camera available under ₹80,000 is {best_cam}/10 → {'✓ match' if got_cam == best_cam else '✗ MISMATCH'}")

## 7. Free-text requests (the AI layer, offline)

A shopper who isn't one of our four personas can just type what they want. `nlp_parse.parse()` turns free text into the same `{weights, budget_max, form_factor}` contract a persona provides — using Gemini when a key is present, and a deterministic keyword/regex parser otherwise.

**The demo never depends on a network call:** the cells below run on the rule-based path with no key and no connectivity.

In [ ]:
from src.recommender import config, nlp_parse

print("GEMINI_ENABLED:", config.GEMINI_ENABLED, "→ using", "Gemini (with fallback)" if config.GEMINI_ENABLED else "the deterministic parser")

for request in ["photography under ₹50k", "gaming phone with big battery below 30k", "a compact flagship", "asdf qwerty"]:
    parsed = nlp_parse.parse(request)
    top3 = wsm.recommend(df, parsed["weights"], budget_max=parsed["budget_max"],
                         form_factor=parsed["form_factor"], top_n=3)
    print("─" * 100)
    print(f'"{request}"')
    print(f"  parsed → weights={parsed['weights']} budget={parsed['budget_max']} form={parsed['form_factor']}")
    print("  top-3 →", ", ".join(f"{r.model_name} (₹{r.price_inr:,})" for r in top3.itertuples()))
print("─" * 100)
print('\nNote: gibberish → neutral 0.25 weights and no crash. "photography under ₹50k" returns the best')
print('cameras that actually exist under ₹50,000 (the A-series) — not a 200MP flagship, because per')
print("Finding 1 none exists below ₹1,39,999. The model is honest about what the catalog can offer.")

## 8. Wrap-up

### The three findings
1. **The segments don't overlap on price, and "photography" is a luxury tier** — every photography-segment phone is ≥ ₹1,39,999 while budget caps at ₹21,999. A camera-focused shopper on a modest budget can't be served the marketing answer, so we rank on measured capability within budget instead.
2. **Cheap phones have the biggest batteries** — price correlates *negatively* with capacity (budget ≈ 5,500 mAh vs business ≈ 4,618 mAh). Battery is the one axis where a budget shopper compromises nothing.
3. **Megapixels are useless as a camera signal here** — only two distinct MP values exist across 36 phones, and every 200MP phone is ≥ ₹1.4 lakh. This one broke our first model and forced us to rebuild `camera_score` around optics tier.

### Limitations (stated plainly)
- **The scores are heuristics, not benchmarks.** `camera_score` encodes our judgement about series optics; it is not DxOMark. `performance_score` is a chipset tier table, not a measured benchmark run. They are defensible and transparent — not authoritative.
- **The 9 phones from 2026 are projections, not facts.** They are flagged `spec_source=mock` everywhere, including in the recommendation text itself, and they *do* win some personas — a demo viewer must read those as "if Samsung ships roughly this, it would rank here."
- **`value_score` is price-driven by construction**, so it structurally favours budget phones. That's intended, but it means value weight is powerful and shouldn't be raised casually.
- **The catalog is 36 curated models**, one row per model — no storage/colour variants, no regional pricing, no discounts. Real MRP moves constantly.
- **A weighted sum assumes the four factors trade off linearly and independently.** In reality they don't (a big battery costs thickness; a big screen costs battery life). It's the right model for an explainable prototype, not a claim about how phones truly work.